In [ ]:
from pathlib import Path
import pandas as pd

path_x = Path("../data/raw/X.csv")
df_x = pd.read_csv(path_x, sep=",", decimal=".", encoding="utf-8-sig")
path_y = Path("../data/raw/Y.csv")
df_y = pd.read_csv(path_y, sep=";", decimal=",", encoding="utf-8-sig")
path_z = Path("../data/raw/Z.csv")
df_z = pd.read_csv(path_z, sep=",", decimal=".", encoding="utf-8-sig")

df_x["Price"] = df_x["Price"].astype(float)
df_y["Price"] = df_y["Price"].astype(float)
df_z["Price"] = df_z["Price"].astype(float)
df_x["Date"] = pd.to_datetime(df_x["Date"], format='%Y-%m-%d')
df_y["Date"] = pd.to_datetime(df_y["Date"], dayfirst=True)
df_z["Date"] = pd.to_datetime(df_z["Date"], format='%Y-%m-%d')

df_x = df_x.rename(columns={"Price": "Price_X"})
df_y = df_y.rename(columns={"Price": "Price_Y"})
df_z = df_z.rename(columns={"Price": "Price_Z"})

df = df_x.merge(df_y, on="Date", how="outer").merge(df_z, on="Date", how="outer")
df.dropna(inplace=True)
df = df.sort_values("Date").reset_index(drop=True)

path_h = Path("../data/raw/historico_equipos.csv")
df_h = pd.read_csv(path_h, sep=",", decimal=".", encoding="utf-8-sig")
df_h["Date"] = pd.to_datetime(df_h["Date"], format='%Y-%m-%d')


fechas_iguales = df["Date"].equals(df_h["Date"])
print(f"{'Fechas iguales' if fechas_iguales else 'Fechas diferentes'} - len_df: {len(df)}, len_df_h: {len(df_h)}")

if not fechas_iguales:
    solo_mio = set(df["Date"]) - set(df_h["Date"])
    solo_hist = set(df_h["Date"]) - set(df["Date"])
    print(len(solo_mio), "solo en mi unión", len(solo_hist), "solo en histórico")

m = df.merge(df_h[["Date", "Price_X", "Price_Y", "Price_Z"]], on="Date", suffixes=("", "_h"))
for c in ["Price_X", "Price_Y", "Price_Z"]:
    ok = (m[c] - m[f"{c}_h"]).abs() < 0.01
    print(c, f"{ok.mean():.1%} coinciden")

Date             datetime64[us]
Price_X                 float64
Price_Y                 float64
Price_Z                 float64
Price_Equipo1           float64
Price_Equipo2           float64
dtype: object